In [1]:
import ollama
import cv2
import json
import re
import os
import shutil

from pydantic import BaseModel, ValidationError
from typing import List, Optional

In [2]:
REVIEW_FOLDER = "reviews"
os.makedirs(REVIEW_FOLDER, exist_ok=True)

PROCESSED_FOLDER = "processed"
os.makedirs(PROCESSED_FOLDER, exist_ok=True)

In [3]:
def preprocess_image(image_path):
    image = cv2.imread(image_path)

    # Guard: return None if image could not be loaded
    if image is None:
        print(f"Warning: Could not read image: {image_path}")
        return None

    # Resize
    image = cv2.resize(image, None, fx=2, fy=2)

    # Denoise
    image = cv2.fastNlMeansDenoisingColored(image, None, 10, 10, 7, 21)

    # Construct the path to save inside PROCESSED_FOLDER
    filename = f"processed_{os.path.basename(image_path)}"
    processed_path = os.path.join(PROCESSED_FOLDER, filename)

    cv2.imwrite(processed_path, image)

    return processed_path

In [4]:
PROMPT = """
You are an information extraction system.

Extract the following fields from the invoice image:

{
  "title": string | null,
  "tax": number | null,
  "expenses": number,
  "date": string | null,
  "items": list[string]
}

CRITICAL DATE RULE:
If the date on the invoice is "12/05/24", "May 12, 2024", or "12-05-2024", 
you MUST convert it to "2024-05-12" assume this for all types of date it should always return as year/month/date. 
If the year is missing, assume 2026.


STRICT RULES:
- Return ONLY valid JSON
- Do NOT guess values
- If unsure, return null
- expenses must be FINAL TOTAL
- tax must be numeric
- date format: YYYY-MM-DD
- items must be short descriptions


"""

In [5]:
def call_qwen(image_path):
    response = ollama.chat(
        model='qwen3.5:4b',
        messages=[{
            'role': 'user',
            'content': PROMPT,
            'images': [image_path]
        }],
        format='json'
    )

    return response['message']['content']

In [6]:
class Invoice(BaseModel):
    title: Optional[str]
    tax: Optional[float]
    expenses: float
    date: Optional[str]
    items: List[str]

In [7]:
def validate_json(response_text):
    try:
        data = json.loads(response_text)
        return Invoice.model_validate(data) # Modern Pydantic v2 method
    except (json.JSONDecodeError, ValidationError):
        return None

In [8]:
def business_validation(data: Invoice):
    if data.expenses <= 0:
        return False

    if data.tax is not None and data.tax < 0:
        return False

    if data.date:
        if not re.match(r"\d{4}-\d{2}-\d{2}", data.date):
            return False

    return True

In [9]:
def extract_with_retry(image_path, max_retries=2):
    for attempt in range(max_retries):

        result = call_qwen(image_path)

        validated = validate_json(result)

        if validated and business_validation(validated):
            print(f"Success on attempt {attempt+1}")
            return validated

        print(f"Retrying... attempt {attempt+1}")

    return None

In [10]:
def verify_output(image_path, data: Invoice):
    checks = []

    # Check 1: tax + some subtotal should roughly equal expenses (if tax present)
    if data.tax is not None and data.items:
        # If tax is a percentage-like value (0-100), skip this check
        if data.tax > 1:  # treat as absolute amount
            implied_subtotal = data.expenses - data.tax
            checks.append(implied_subtotal > 0)

    # Check 2: item count is reasonable
    checks.append(0 < len(data.items) <= 50)

    # Check 3: expenses is a plausible monetary value
    checks.append(0 < data.expenses < 1_000_000)

    # Check 4: date is a valid calendar date
    if data.date:
        try:
            from datetime import datetime
            datetime.strptime(data.date, "%Y-%m-%d")
            checks.append(True)
        except ValueError:
            checks.append(False)

    return all(checks) if checks else False

In [11]:
def compute_confidence(data, verified):
    score = 0
    possible = 0

    # -------- Expenses (VERY IMPORTANT) --------
    possible += 0.4
    if data.expenses and data.expenses > 0:
        score += 0.4

    # -------- Tax --------
    possible += 0.1
    if data.tax is not None:
        score += 0.1

    # -------- Date --------
    possible += 0.15
    if data.date:
        if re.match(r"\d{4}-\d{2}-\d{2}", data.date):
            score += 0.15

    # -------- Title --------
    possible += 0.1
    if data.title and len(data.title.strip()) > 2:
        score += 0.1

    # -------- Items --------
    possible += 0.15
    if data.items and len(data.items) > 0:
        score += 0.15

    # -------- Verification --------
    possible += 0.1
    if verified:
        score += 0.1

    return round(score / possible, 2) if possible else 0.0

In [12]:
def send_to_review(image_path, result=None):
    filename = os.path.basename(image_path)
    destination = os.path.join(REVIEW_FOLDER, filename)

    shutil.copy(image_path, destination)

    # Save JSON if available
    if result:
        base = os.path.splitext(filename)[0]
        with open(os.path.join(REVIEW_FOLDER, base + ".json"), "w") as f:
            f.write(result)

    print(f"Sent to review: {destination}")

In [13]:
def process_invoice(image_path):
    # Step 1: preprocess
    processed_img = preprocess_image(image_path)

    # Guard: if preprocessing failed (unreadable image), send to review
    if processed_img is None:
        send_to_review(image_path)
        return {"status": "failed", "message": "Could not preprocess image. Sent to review"}

    # Step 2: extraction
    result = extract_with_retry(processed_img)

    # Guard: extraction failed after all retries
    if not result:
        send_to_review(image_path)
        return {"status": "failed", "message": "Extraction failed. Sent to review"}

    # Step 3: verification
    result_json = result.model_dump_json()
    verified = verify_output(processed_img, result)

    # Step 4: confidence
    confidence = compute_confidence(result, verified)

    # Step 5: decision — low confidence OR failed verification → review
    if confidence < 0.85 or not verified:
        send_to_review(image_path, result_json)
        return {"status": "low_confidence", "confidence": confidence, "verified": verified}

    return {
        "status": "success",
        "confidence": confidence,
        "data": result.model_dump()
    }

In [14]:
result = process_invoice(r"C:\Users\r1005\.cache\kagglehub\datasets\osamahosamabdellatif\high-quality-invoice-images-for-ocr\versions\3\batch_1\batch_1\batch1_1\batch1-0001.jpg")
print(result)

Success on attempt 1
{'status': 'success', 'confidence': 1.0, 'data': {'title': 'Invoice no: 51109338', 'tax': 564.02, 'expenses': 6204.19, 'date': '2013-04-13', 'items': ['CLEARANCE! Fast Dell Desktop Computer PC DUAL CORE WINDOWS 10 4/8/16GB RAM', 'HP T520 Thin Client Computer AMD GX-212JC 1.2GHz 4GB RAM TESTED !!READ BELOW!!', 'gaming pc desktop computer', '12-Core Gaming Computer Desktop PC Tower Affordable GAMING PC 8GB AMD Vega RGB', 'Custom Build Dell Optiplex 9020 MT i5-4570 3.20GHz Desktop Computer PC', 'Dell Optiplex 990 MT Computer PC Quad Core i7 3.4GHz 16GB 2TB HD Windows 10 Pro', 'Dell Core 2 Duo Desktop Computer | Windows XP Pro | 4GB | 500GB']}}


In [15]:
result = process_invoice(r"C:\Users\r1005\Documents\richard whitebg.png")
print(result)

Retrying... attempt 1
Retrying... attempt 2
Sent to review: reviews\richard whitebg.png
{'status': 'failed', 'message': 'Extraction failed. Sent to review'}


In [ ]:
import os

def process_target(target_path):
    # Define what file types we actually want to process
    valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff')
    
    all_results = []

    # Scenario A: The path is a single file
    if os.path.isfile(target_path):
        if target_path.lower().endswith(valid_extensions):
            print(f"Processing single file: {target_path}")
            result = process_invoice(target_path)
            all_results.append({"file": target_path, "result": result})
        else:
            print(f"Skipped {target_path} (not a supported image format).")

    # Scenario B: The path is a folder
    elif os.path.isdir(target_path):
        print(f"Scanning directory: {target_path}")
        for filename in os.listdir(target_path):
            if filename.lower().endswith(valid_extensions):
                full_path = os.path.join(target_path, filename)
                print(f"Processing: {filename}")
                
                result = process_invoice(full_path)
                all_results.append({"file": full_path, "result": result})
    
    # Scenario C: Invalid path
    else:
        print(f"Error: The path '{target_path}' does not exist.")

    return all_results

In [ ]:
# Pass the folder path containing multiple images
target = r"C:\Users\r1005\.cache\kagglehub\datasets\osamahosamabdellatif\high-quality-invoice-images-for-ocr\versions\3\batch_1\batch_1\batch1_1"

results = process_target(target)

# Print a summary of all processed files
for item in results:
    print(f"\nFile: {item['file']}")
    print(f"Status: {item['result']['status']}")

In [ ]:
# Pass a direct file path
target = r"C:\Users\r1005\.cache\kagglehub\datasets\osamahosamabdellatif\high-quality-invoice-images-for-ocr\versions\3\batch_1\batch_1\batch1_1\batch1-0001.jpg"

results = process_target(target)
print(results)
